# CSIC 2010 — Data Preprocessing

Parse raw HTTP request files (`.txt`) to structured CSV.

**Inputs:**
- `anomalousTrafficTest.txt` — anomalous requests
- `normalTrafficTraining.txt` — normal requests

**Output:** `../../data/raw/d3_csic2010_raw.csv` (theo task 13/07 — Bách)

In [9]:
import pandas as pd
import re
from pathlib import Path

## Cell 1: Định nghĩa hàm parse HTTP requests

In [10]:
def _parse_request_line(line: str):
    m = re.match(r'^(GET|POST|PUT|DELETE|HEAD|OPTIONS)\s+(\S+)\s+HTTP/\S+', line)
    if not m:
        return None
    method, url = m.groups()
    parsed = re.match(r'https?://[^/]+(\S*)', url)
    path_full = parsed.group(1) if parsed else url
    if '?' in path_full:
        path, query = path_full.split('?', 1)
    else:
        path, query = path_full, ''
    return {'method': method, 'path': path, 'query': query}


def parse_http_requests(raw_text: str) -> pd.DataFrame:
    lines = raw_text.split('\n')
    records = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()
        req = _parse_request_line(line)
        if not req:
            i += 1
            continue

        method = req['method']
        path = req['path']
        query = req['query']
        raw_lines = [line]

        # Parse headers
        headers = {}
        i += 1
        while i < n:
            line = lines[i]
            raw_lines.append(line)
            if line.strip() == '':
                i += 1
                break
            if ':' in line:
                k, v = line.split(':', 1)
                headers[k.strip().lower()] = v.strip()
            i += 1

        # Remaining lines are body until next request
        body_lines = []
        while i < n:
            line = lines[i]
            if _parse_request_line(line.strip()):
                break
            body_lines.append(line)
            i += 1

        body = ''.join(body_lines).strip()
        raw_req = '\n'.join(raw_lines)
        if body:
            raw_req += '\n' + body

        records.append({
            'method': method,
            'path': path,
            'query': query,
            'host': headers.get('host', ''),
            'cookie': headers.get('cookie', ''),
            'body': body,
            'raw_request': raw_req
        })

    return pd.DataFrame(records)

## Cell 2: Load & parse anomalous + normal

In [11]:
base = Path.cwd()
print(f"Working dir: {base}")

# Load anomalous
txt_anom = Path("anomalousTrafficTest.txt").read_text(encoding="utf-8")
df_anom = parse_http_requests(txt_anom)
df_anom['label'] = 'anomalous'
print(f"Anomalous: {len(df_anom)} requests")

# Load normal
txt_norm = Path("normalTrafficTraining.txt").read_text(encoding="utf-8")
df_norm = parse_http_requests(txt_norm)
df_norm['label'] = 'normal'
print(f"Normal: {len(df_norm)} requests")

Working dir: d:\Research\AISecurity\sql-injection-continual-leanrning\AI-Based-SQL-Injection-Detection-System\notebooks\cisc-2010-data-pre-process
Anomalous: 25065 requests
Normal: 36000 requests


## Cell 3: Gộp & in thông tin

In [12]:
df = pd.concat([df_anom, df_norm], ignore_index=True)
print(f"Total: {len(df)} requests")
print()
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print()
print("Label distribution:")
print(df['label'].value_counts())
print()
print("Method distribution:")
print(df['method'].value_counts())

Total: 61065 requests

Shape: (61065, 8)
Columns: ['method', 'path', 'query', 'host', 'cookie', 'body', 'raw_request', 'label']
dtypes:
method         str
path           str
query          str
host           str
cookie         str
body           str
raw_request    str
label          str
dtype: object

Label distribution:
label
normal       36000
anomalous    25065
Name: count, dtype: int64

Method distribution:
method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64


In [13]:
print("Sample rows (anomalous):")
df[df['label'] == 'anomalous'][['method', 'path', 'query']].head(3)
print()
print("Sample rows (normal):")
df[df['label'] == 'normal'][['method', 'path', 'query']].head(3)

Sample rows (anomalous):

Sample rows (normal):


,method,path,query
25065,GET,/tienda1/index.jsp,
25066,GET,/tienda1/publico/anadir.jsp,id=3&nombre=Vino+Rioja&precio=100&cantidad=55&...
25067,POST,/tienda1/publico/anadir.jsp,


## Cell 4: Xuất CSV — target theo task 13/07

Thư mục target: `data/raw/d3_csic2010_raw.csv`

In [14]:
output_dir = Path("../../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "d3_csic2010_raw.csv"

df.to_csv(output_path, index=False)
print(f"Exported {len(df)} rows → {output_path.resolve()}")

Exported 61065 rows → D:\Research\AISecurity\sql-injection-continual-leanrning\AI-Based-SQL-Injection-Detection-System\data\raw\d3_csic2010_raw.csv


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   method       61065 non-null  str  
 1   path         61065 non-null  str  
 2   query        61065 non-null  str  
 3   host         61065 non-null  str  
 4   cookie       61065 non-null  str  
 5   body         61065 non-null  str  
 6   raw_request  61065 non-null  str  
 7   label        61065 non-null  str  
dtypes: str(8)
memory usage: 3.7 MB
